## Imports & Path Setup

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np


PROJECT_ROOT = Path.cwd().parents[1]

sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = (
    PROJECT_ROOT
    / "src"
    / "data"
    / "raw"
    / "SPY"
    / "2010"
    / "spy_eod_201001.txt"
)

## Load Data

In [2]:
from src.data.ingestion import load_raw_option_chain

df = load_raw_option_chain(DATA_PATH)

print(df.shape)
df.head()

(18668, 33)


,quote_unixtime,quote_readtime,quote_date,quote_time_hours,underlying_last,expire_date,expire_unix,dte,c_delta,c_gamma,...,p_last,p_delta,p_gamma,p_vega,p_theta,p_rho,p_iv,p_volume,strike_distance,strike_distance_pct
0,1262638800,2010-01-04 16:00:00,2010-01-04,16.0,113.29,2010-01-15,1263589200,11.0,0.88304,0.00005,...,0.02,-0.00143,0.00010,0.00074,-0.00435,-0.00050,1.36742,0.0,58.3,0.515
1,1262638800,2010-01-04 16:00:00,2010-01-04,16.0,113.29,2010-01-15,1263589200,11.0,0.88398,0.00005,...,0.00,-0.00160,0.00020,0.00074,-0.00396,-0.00043,1.33476,NaN,57.3,0.506
2,1262638800,2010-01-04 16:00:00,2010-01-04,16.0,113.29,2010-01-15,1263589200,11.0,0.88657,0.00000,...,0.03,-0.00146,0.00014,0.00106,-0.00369,-0.00004,1.30655,0.0,56.3,0.497
3,1262638800,2010-01-04 16:00:00,2010-01-04,16.0,113.29,2010-01-15,1263589200,11.0,0.88750,0.00003,...,0.04,-0.00166,0.00017,0.00056,-0.00428,-0.00051,1.27237,0.0,55.3,0.488
4,1262638800,2010-01-04 16:00:00,2010-01-04,16.0,113.29,2010-01-15,1263589200,11.0,0.88970,0.00007,...,0.04,-0.00167,0.00016,0.00128,-0.00429,-0.00023,1.24421,0.0,54.3,0.479


## Construct Time-to-Expiry

In [3]:
SECONDS_PER_YEAR = 365.25 * 24 * 60 * 60

df["t_years"] = (
    df["expire_unix"] - df["quote_unixtime"]
) / SECONDS_PER_YEAR

df["t_years"].describe()

count    18668.000000
mean         0.589280
std          0.577620
min          0.000000
25%          0.175108
50%          0.437942
75%          0.892539
max          2.962355
Name: t_years, dtype: float64

## Basic Undiscounted Bounds
Without any interest rates or dividends (r = q = 0):
  - **Call:** $max(0, S_0-K) \leq C \leq S_0$
  - **Put:** $max(0, K-S_0) \leq P \leq K$

In [ ]:
# Theorectical upper and lower bounds for call
df["call_lower_bound"] = np.maximum(
    0.0,
    df["underlying_last"] - df["strike"]
)
df["call_upper_bound"] = df["underlying_last"]

# Theorectical upper and lower bounds for put
df["put_lower_bound"] = np.maximum(
    0.0,
    df["strike"] - df["underlying_last"]
)
df["put_upper_bound"] = df["strike"]

## Identify Violations

In [5]:
# Define condition
call_valid = df["c_last"].notna()
put_valid = df["p_last"].notna()

# Lower and upper violations for call
call_lower_violation = (
    call_valid &
    (df["c_last"] < df["call_lower_bound"])
)
call_upper_violation = (
    call_valid &
    (df["c_last"] > df["call_upper_bound"])
)

# Lower and upper violations for put
put_lower_violation = (
    put_valid &
    (df["p_last"] < df["put_lower_bound"])
)
put_upper_violation = (
    put_valid &
    (df["p_last"] > df["put_upper_bound"])
)

In [6]:
print("Call lower-bound violations:", call_lower_violation.sum())
print("Call upper-bound violations:", call_upper_violation.sum())
print("Put lower-bound violations:", put_lower_violation.sum())
print("Put upper-bound violations:", put_upper_violation.sum())

Call lower-bound violations: 6004
Call upper-bound violations: 0
Put lower-bound violations: 2100
Put upper-bound violations: 0


## Inspect Violations

In [8]:
# Call Violations
df.loc[
    call_lower_violation,
    [
        "quote_readtime",
        "underlying_last",
        "strike",
        "dte",
        "c_bid",
        "c_ask",
        "c_last"
    ],
].head(20)

,quote_readtime,underlying_last,strike,dte,c_bid,c_ask,c_last
0,2010-01-04 16:00:00,113.29,55.0,11.0,58.20,58.40,54.67
1,2010-01-04 16:00:00,113.29,56.0,11.0,57.20,57.40,53.30
2,2010-01-04 16:00:00,113.29,57.0,11.0,56.20,56.39,0.00
3,2010-01-04 16:00:00,113.29,58.0,11.0,55.19,55.40,0.00
4,2010-01-04 16:00:00,113.29,59.0,11.0,54.20,54.40,0.00
5,2010-01-04 16:00:00,113.29,60.0,11.0,53.20,53.40,52.30
6,2010-01-04 16:00:00,113.29,61.0,11.0,52.20,52.40,51.30
7,2010-01-04 16:00:00,113.29,62.0,11.0,51.19,51.40,50.30
8,2010-01-04 16:00:00,113.29,63.0,11.0,50.20,50.40,49.30
9,2010-01-04 16:00:00,113.29,64.0,11.0,49.20,49.39,0.00


In [ ]:
# Put Violations
df.loc[
    put_lower_violation,
    [
        "quote_readtime",
        "underlying_last",
        "strike",
        "dte",
        "p_bid",
        "p_ask",
        "p_last"
    ],
].head(20)

,quote_readtime,underlying_last,strike,dte,c_bid,c_ask,c_last
86,2010-01-04 16:00:00,113.29,141.0,11.0,0.00,0.01,0.00
89,2010-01-04 16:00:00,113.29,144.0,11.0,0.00,0.03,0.00
94,2010-01-04 16:00:00,113.29,149.0,11.0,0.00,0.02,0.00
95,2010-01-04 16:00:00,113.29,150.0,11.0,0.00,0.01,0.00
181,2010-01-04 16:00:00,113.29,127.0,46.0,0.02,0.04,0.03
182,2010-01-04 16:00:00,113.29,128.0,46.0,0.01,0.03,0.03
183,2010-01-04 16:00:00,113.29,129.0,46.0,0.01,0.05,0.03
190,2010-01-04 16:00:00,113.29,136.0,46.0,0.00,0.03,0.00
191,2010-01-04 16:00:00,113.29,137.0,46.0,0.00,0.03,0.01
192,2010-01-04 16:00:00,113.29,138.0,46.0,0.00,0.03,0.00


## Calculate & Test Midpoint Prices

In [10]:
# Calculate midpoint
df["c_mid"] = (df["c_bid"] + df["c_ask"]) / 2
df["p_mid"] = (df["p_bid"] + df["p_ask"]) / 2

df[
    [
        "underlying_last",
        "strike",
        "dte",
        "c_bid",
        "c_ask",
        "c_mid",
        "c_last",
        "p_bid",
        "p_ask",
        "p_mid",
        "p_last",
    ]
].head(20)

,underlying_last,strike,dte,c_bid,c_ask,c_mid,c_last,p_bid,p_ask,p_mid,p_last
0,113.29,55.0,11.0,58.20,58.40,58.300,54.67,0.0,0.02,0.010,0.02
1,113.29,56.0,11.0,57.20,57.40,57.300,53.30,0.0,0.01,0.005,0.00
2,113.29,57.0,11.0,56.20,56.39,56.295,0.00,0.0,0.01,0.005,0.03
3,113.29,58.0,11.0,55.19,55.40,55.295,0.00,0.0,0.03,0.015,0.04
4,113.29,59.0,11.0,54.20,54.40,54.300,0.00,0.0,0.02,0.010,0.04
5,113.29,60.0,11.0,53.20,53.40,53.300,52.30,0.0,0.02,0.010,0.04
6,113.29,61.0,11.0,52.20,52.40,52.300,51.30,0.0,0.02,0.010,0.00
7,113.29,62.0,11.0,51.19,51.40,51.295,50.30,0.0,0.02,0.010,0.08
8,113.29,63.0,11.0,50.20,50.40,50.300,49.30,0.0,0.02,0.010,0.00
9,113.29,64.0,11.0,49.20,49.39,49.295,0.00,0.0,0.02,0.010,0.00


In [11]:
# Test midpoint
call_mid_lower_violation = (
    df["c_mid"].notna()
    & (df["c_mid"] < df["call_lower_bound"])
)

call_mid_upper_violation = (
    df["c_mid"].notna()
    & (df["c_mid"] > df["call_upper_bound"])
)

put_mid_lower_violation = (
    df["p_mid"].notna()
    & (df["p_mid"] < df["put_lower_bound"])
)

put_mid_upper_violation = (
    df["p_mid"].notna()
    & (df["p_mid"] > df["put_upper_bound"])
)

print("Call midpoint lower-bound violations:",
      call_mid_lower_violation.sum())

print("Call midpoint upper-bound violations:",
      call_mid_upper_violation.sum())

print("Put midpoint lower-bound violations:",
      put_mid_lower_violation.sum())

print("Put midpoint upper-bound violations:",
      put_mid_upper_violation.sum())

Call midpoint lower-bound violations: 562
Call midpoint upper-bound violations: 0
Put midpoint lower-bound violations: 639
Put midpoint upper-bound violations: 0


## Compare Last vs Midpoint Prices

In [12]:
print("Call last-price lower violations:",
      call_lower_violation.sum())

print("Call midpoint lower violations:",
      call_mid_lower_violation.sum())

print()

print("Put last-price lower violations:",
      put_lower_violation.sum())

print("Put midpoint lower violations:",
      put_mid_lower_violation.sum())

Call last-price lower violations: 6004
Call midpoint lower violations: 562

Put last-price lower violations: 2100
Put midpoint lower violations: 639


## Inspect Call & Put Midpoint Violations

In [13]:
# Call
df.loc[
    call_mid_lower_violation,
    [
        "quote_readtime",
        "underlying_last",
        "strike",
        "dte",
        "c_bid",
        "c_ask",
        "c_mid",
        "c_last",
    ],
].head(20)

,quote_readtime,underlying_last,strike,dte,c_bid,c_ask,c_mid,c_last
110,2010-01-04 16:00:00,113.29,56.0,46.00,57.15,57.41,57.280,53.77
114,2010-01-04 16:00:00,113.29,60.0,46.00,53.14,53.44,53.290,0.00
417,2010-01-04 16:00:00,113.29,59.0,164.96,54.14,54.44,54.290,33.20
726,2010-01-04 16:00:00,113.29,45.0,347.00,68.05,68.40,68.225,0.00
760,2010-01-04 16:00:00,113.29,50.0,361.00,63.10,63.45,63.275,56.85
839,2010-01-04 16:00:00,113.29,20.0,711.00,93.05,93.45,93.250,92.37
840,2010-01-04 16:00:00,113.29,25.0,711.00,88.05,88.45,88.250,80.70
841,2010-01-04 16:00:00,113.29,30.0,711.00,83.05,83.50,83.275,75.55
914,2010-01-05 16:00:00,113.64,55.0,10.00,58.56,58.69,58.625,54.67
915,2010-01-05 16:00:00,113.64,56.0,10.00,57.55,57.69,57.620,53.30


In [14]:
# Puts
df.loc[
    put_mid_lower_violation,
    [
        "quote_readtime",
        "underlying_last",
        "strike",
        "dte",
        "p_bid",
        "p_ask",
        "p_mid",
        "p_last",
    ],
].head(20)

,quote_readtime,underlying_last,strike,dte,p_bid,p_ask,p_mid,p_last
68,2010-01-04 16:00:00,113.29,123.0,11.0,9.60,9.80,9.700,11.02
69,2010-01-04 16:00:00,113.29,124.0,11.0,10.59,10.80,10.695,13.99
70,2010-01-04 16:00:00,113.29,125.0,11.0,11.60,11.79,11.695,12.36
71,2010-01-04 16:00:00,113.29,126.0,11.0,12.60,12.79,12.695,16.10
72,2010-01-04 16:00:00,113.29,127.0,11.0,13.60,13.81,13.705,18.30
73,2010-01-04 16:00:00,113.29,128.0,11.0,14.60,14.80,14.700,17.55
74,2010-01-04 16:00:00,113.29,129.0,11.0,15.60,15.80,15.700,18.49
75,2010-01-04 16:00:00,113.29,130.0,11.0,16.61,16.80,16.705,16.75
76,2010-01-04 16:00:00,113.29,131.0,11.0,17.59,17.80,17.695,20.20
77,2010-01-04 16:00:00,113.29,132.0,11.0,18.60,18.81,18.705,21.20
